In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import joblib




In [2]:
data = pd.read_csv('supply_chain_data.csv')
print(data.head())

  Product type   SKU      Price  Availability  Number of products sold  \
0     haircare  SKU0  69.808006            55                      802   
1     skincare  SKU1  14.843523            95                      736   
2     haircare  SKU2  11.319683            34                        8   
3     skincare  SKU3  61.163343            68                       83   
4     skincare  SKU4   4.805496            26                      871   

   Revenue generated Customer demographics  Stock levels  Lead times  \
0        8661.996792            Non-binary            58           7   
1        7460.900065                Female            53          30   
2        9577.749626               Unknown             1          10   
3        7766.836426            Non-binary            23          13   
4        2686.505152            Non-binary             5           3   

   Order quantities  ...  Location Lead time  Production volumes  \
0                96  ...    Mumbai        29          

In [3]:
data.columns
data.info()
data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product type             100 non-null    str    
 1   SKU                      100 non-null    str    
 2   Price                    100 non-null    float64
 3   Availability             100 non-null    int64  
 4   Number of products sold  100 non-null    int64  
 5   Revenue generated        100 non-null    float64
 6   Customer demographics    100 non-null    str    
 7   Stock levels             100 non-null    int64  
 8   Lead times               100 non-null    int64  
 9   Order quantities         100 non-null    int64  
 10  Shipping times           100 non-null    int64  
 11  Shipping carriers        100 non-null    str    
 12  Shipping costs           100 non-null    float64
 13  Supplier name            100 non-null    str    
 14  Location                 100 non-null 

,Price,Availability,Number of products sold,Revenue generated,Stock levels,Lead times,Order quantities,Shipping times,Shipping costs,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Defect rates,Costs
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000
mean,49.462461,48.400000,460.990000,5776.048187,47.770000,15.960000,49.220000,5.750000,5.548149,17.080000,567.840000,14.77000,47.266693,2.277158,529.245782
std,31.168193,30.743317,303.780074,2732.841744,31.369372,8.785801,26.784429,2.724283,2.651376,8.846251,263.046861,8.91243,28.982841,1.461366,258.301696
min,1.699976,1.000000,8.000000,1061.618523,0.000000,1.000000,1.000000,1.000000,1.013487,1.000000,104.000000,1.00000,1.085069,0.018608,103.916248
25%,19.597823,22.750000,184.250000,2812.847151,16.750000,8.000000,26.000000,3.750000,3.540248,10.000000,352.000000,7.00000,22.983299,1.009650,318.778455
50%,51.239831,43.500000,392.500000,6006.352023,47.500000,17.000000,52.000000,6.000000,5.320534,18.000000,568.500000,14.00000,45.905622,2.141863,520.430444
75%,77.198228,75.000000,704.250000,8253.976921,73.000000,24.000000,71.250000,8.000000,7.601695,25.000000,797.000000,23.00000,68.621026,3.563995,763.078231
max,99.171329,100.000000,996.000000,9866.465458,100.000000,30.000000,96.000000,10.000000,9.929816,30.000000,985.000000,30.00000,99.466109,4.939255,997.413450


In [4]:
data.head()


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [5]:
data.isnull().sum()

Product type               0
SKU                        0
Price                      0
Availability               0
Number of products sold    0
Revenue generated          0
Customer demographics      0
Stock levels               0
Lead times                 0
Order quantities           0
Shipping times             0
Shipping carriers          0
Shipping costs             0
Supplier name              0
Location                   0
Lead time                  0
Production volumes         0
Manufacturing lead time    0
Manufacturing costs        0
Inspection results         0
Defect rates               0
Transportation modes       0
Routes                     0
Costs                      0
dtype: int64

In [6]:
data.dropna()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,haircare,SKU95,77.903927,65,672,7386.363944,Unknown,15,14,26,...,Mumbai,18,450,26,58.890686,Pending,1.210882,Air,Route A,778.864241
96,cosmetics,SKU96,24.423131,29,324,7698.424766,Non-binary,67,2,32,...,Mumbai,28,648,28,17.803756,Pending,3.872048,Road,Route A,188.742141
97,haircare,SKU97,3.526111,56,62,4370.916580,Male,46,19,4,...,Mumbai,10,535,13,65.765156,Fail,3.376238,Road,Route A,540.132423
98,skincare,SKU98,19.754605,43,913,8525.952560,Female,53,1,27,...,Chennai,28,581,9,5.604691,Pending,2.908122,Rail,Route A,882.198864


In [7]:
## Target and Features
target = 'Number of products sold'
features = data.columns.drop(target)

In [8]:
X = data[features]
y = data[target]

In [9]:
##Define columns

In [10]:
categorical_cols = ['Product type', 'Customer demographics', 'Shipping carriers', 'Supplier name', 'Location', 'Inspection results', 'Transportation modes', 'Routes']   

In [11]:
numerical_cols = ["Price", "Lead time", "Availability", "Stock levels", "Inventory levels", "Demand forecasts", "Production schedules", "Order quantities", "Delivery times", "Costs"]

In [12]:
##Split raw data before preprocessing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [13]:
##Preprocess for Neural Network
nn_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Use numeric columns that actually exist in the dataset
numerical_cols = X_train_raw.select_dtypes(include=np.number).columns.tolist()

# Rebuild the preprocessor with the corrected columns
nn_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

X_train_nn = nn_preprocessor.fit_transform(X_train_raw)
X_test_nn = nn_preprocessor.transform(X_test_raw)

y_scaler = StandardScaler()
y_train_nn = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
y_test_nn = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).ravel()


In [14]:
import json
print(list(X_train_raw.columns))
print(json.dumps(X_train_raw.iloc[0].to_dict(), indent=2, default=str))

['Product type', 'SKU', 'Price', 'Availability', 'Revenue generated', 'Customer demographics', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location', 'Lead time', 'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Inspection results', 'Defect rates', 'Transportation modes', 'Routes', 'Costs']
{
  "Product type": "haircare",
  "SKU": "SKU55",
  "Price": 79.85505834078943,
  "Availability": 16,
  "Revenue generated": 2925.6751703038126,
  "Customer demographics": "Male",
  "Stock levels": 97,
  "Lead times": 11,
  "Order quantities": 11,
  "Shipping times": 5,
  "Shipping carriers": "Carrier A",
  "Shipping costs": 5.014364955030907,
  "Supplier name": "Supplier 2",
  "Location": "Delhi",
  "Lead time": 27,
  "Production volumes": 918,
  "Manufacturing lead time": 5,
  "Manufacturing costs": 30.3235452566165,
  "Inspection results": "Fail",
  "Defect rates": 4.548919659396385,
  "Transp

In [15]:
##Preprocess for Random Forest and XGBoost
# Keep numeric features in their original units; encode only categorical features.
tree_preprocessor = ColumnTransformer([
    ('num', 'passthrough', numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

X_train_tree = tree_preprocessor.fit_transform(X_train_raw)
X_test_tree = tree_preprocessor.transform(X_test_raw)


In [16]:
##Building Neural Network Model

In [17]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_nn.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])


c:\Users\LENOVO\Desktop\ORACLE BRAIN\Supply chain\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

In [19]:
##Train the Neural Network
history = model.fit(
    X_train_nn, y_train_nn,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)


Epoch 1/100


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - loss: 1.0364 - mae: 0.8656 - val_loss: 0.8495 - val_mae: 0.7967
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.7852 - mae: 0.7543 - val_loss: 0.8573 - val_mae: 0.8264
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.6230 - mae: 0.6753 - val_loss: 0.9388 - val_mae: 0.8607
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.4802 - mae: 0.5873 - val_loss: 1.0266 - val_mae: 0.8820
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.3735 - mae: 0.5080 - val_loss: 1.0972 - val_mae: 0.9155
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.2821 - mae: 0.4342 - val_loss: 1.1596 - val_mae: 0.9420
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.1986 - mae: 0.3607 - val_loss: 1.2202 - val_mae: 0.9571
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.1311 - mae: 0.2925 - val_loss: 1.2918 - val_mae: 0.9671
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0922 - mae: 0.2344 - val

In [20]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Score every model in the original target units.
def to_original_units(predictions):
    return y_scaler.inverse_transform(np.asarray(predictions).reshape(-1, 1)).ravel()

nn_predictions = to_original_units(model.predict(X_test_nn, verbose=0))

random_forest = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
random_forest.fit(X_train_tree, y_train)
rf_predictions = random_forest.predict(X_test_tree)

xgboost_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
xgboost_model.fit(X_train_tree, y_train)
xgb_predictions = xgboost_model.predict(X_test_tree)

results = pd.DataFrame({
    'Model': ['Neural Network', 'Random Forest', 'XGBoost'],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, nn_predictions)),
        np.sqrt(mean_squared_error(y_test, rf_predictions)),
        np.sqrt(mean_squared_error(y_test, xgb_predictions))
    ],
    'R2': [
        r2_score(y_test, nn_predictions),
        r2_score(y_test, rf_predictions),
        r2_score(y_test, xgb_predictions)
    ]
}).sort_values(['RMSE', 'R2'], ascending=[True, False]).reset_index(drop=True)

print(results.to_string(index=False))
best_model = results.iloc[0]
print(
    f"\nBest model: {best_model['Model']} because it has the lowest "
    f"RMSE ({best_model['RMSE']:.2f}) and an R2 of {best_model['R2']:.3f}."
)


         Model       RMSE        R2
 Random Forest 393.457688 -0.622199
       XGBoost 421.986300 -0.865971
Neural Network 469.397253 -1.308815

Best model: Random Forest because it has the lowest RMSE (393.46) and an R2 of -0.622.


In [21]:
import os, joblib

os.makedirs("models", exist_ok=True)

joblib.dump(nn_preprocessor, "models/preprocessor.joblib")
joblib.dump(y_scaler, "models/y_scaler.joblib")
model.save("models/model.keras")